# Cart System — Kaggle CPU benchmark

**READY TO RUN (tooling); benchmark NOT RUN.** Chỉ chuyển EXECUTED khi có artifacts thật. VERIFIED cần review runtime CPU Kaggle, cấu hình backend và artifacts; notebook không tự chứng nhận nguồn đo.

Kaggle là máy phát tải, backend/MySQL/Redis là system under test ở máy khác. Chọn Accelerator=None, bật Internet nếu tài khoản/phiên cho phép. Không tạo tunnel hoặc public database/Redis. Hướng dẫn chi tiết: `docs/kaggle-benchmark.md`.

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json, platform, hashlib
# Đính kèm một Dataset PRIVATE chỉ có scripts sạch, không upload .env/accounts/results.
SUITE_DIR = Path('/kaggle/input/cart-system-load-testing/load-testing')
WORK = Path('/kaggle/working/cart-load-testing')
if sys.version_info < (3, 11):
    raise RuntimeError('Dùng Python >= 3.11; bộ công cụ được kiểm tra local trên 3.12')
if not SUITE_DIR.is_dir():
    raise RuntimeError('Sửa SUITE_DIR theo đường dẫn Dataset thực tế')
WORK.mkdir(parents=True, exist_ok=True)
SCRIPTS = ['requirements.txt', 'tooling.py', 'check_connectivity.py', 'prepare.py',
           'locustfile.py', 'run.py', 'report.py']
for name in SCRIPTS:
    shutil.copy2(SUITE_DIR / name, WORK / name)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WORK / 'requirements.txt')],
               check=True, timeout=600)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
print('Scripts/dependencies sẵn sàng; chưa phát tải.')


## Cấu hình trước Save & Run All

Điền các biến dưới đây bằng cấu hình đã được duyệt. Không dùng input/getpass để tránh kẹt phiên batch. Không đưa secret vào code. Thêm Secret `LOAD_TEST_PASSWORD` qua Add-ons → Secrets và cấp quyền cho notebook. Không cần JWT_SECRET hoặc DB credentials.

PROFILE mặc định smoke. Lần lượt đổi sang low rồi medium/high khi được duyệt; không tự động chạy các mức tải lớn. Backend mô tả phần cứng, image/revision, vùng triển khai và resource limits; unknown nếu chưa biết.

In [ ]:
from tooling import validate_target
TARGET_BASE_URL = ''  # HTTPS origin đã duyệt, không /api, query hoặc credentials
CPU_CONFIRMED = False  # Chỉ đổi True sau khi chọn Accelerator=None trong Kaggle settings
SOURCE_REVISION = ''  # Revision của suite đã upload; script hashes cũng được lưu
BACKEND_DESCRIPTION = 'unknown'
PROFILE = 'smoke'
APPROVED_MAX_USERS = 1  # Giới hạn được chủ backend cho phép, tăng có kiểm soát
if not CPU_CONFIRMED:
    raise RuntimeError('Xác nhận Kaggle Accelerator=None trước khi chạy')
if not SOURCE_REVISION:
    raise RuntimeError('Ghi revision của suite để tái lập phép đo')
TARGET_BASE_URL = validate_target(TARGET_BASE_URL, require_https=True)
os.environ['TARGET_BASE_URL'] = TARGET_BASE_URL
os.environ['LOAD_TEST_REQUIRE_HTTPS'] = '1'
os.environ['LOAD_TEST_TIMEOUT'] = '10'


In [ ]:
# GET-only: DNS public, TLS hợp lệ, không redirect, timeout hữu hạn, kiểm tra product seed.
subprocess.run([sys.executable, str(WORK / 'check_connectivity.py'),
                '--output', str(WORK / 'connectivity.json')], check=True, timeout=40)
print('Connectivity PASS; chưa kiểm chứng login/DB/Redis readiness đầy đủ.')


In [ ]:
# Secrets: fail-closed, không fallback sang prompt ở batch; không in exception nội bộ.
try:
    from kaggle_secrets import UserSecretsClient
    password = UserSecretsClient().get_secret('LOAD_TEST_PASSWORD')
except Exception:
    raise RuntimeError('Không đọc được LOAD_TEST_PASSWORD. Kiểm tra Secret đã attach và Internet access.') from None
if not password:
    raise RuntimeError('LOAD_TEST_PASSWORD không được rỗng')
os.environ['LOAD_TEST_PASSWORD'] = password
del password


In [ ]:
import uuid
from datetime import datetime, timezone
import psutil
from run import PROFILES
if PROFILE not in PROFILES:
    raise ValueError('Profile không hợp lệ')
users = PROFILES[PROFILE][0]
if users > APPROVED_MAX_USERS:
    raise RuntimeError('Profile vượt số users được duyệt')
# Token duy nhất mỗi run để không đè artifacts hoặc dùng chung cart giữa các phiên.
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S') + '-' + uuid.uuid4().hex[:8]
OUTPUT = WORK / 'results' / (PROFILE + '-' + RUN_ID)
OUTPUT.mkdir(parents=True, exist_ok=False)
os.environ['LOAD_TEST_RUN_ID'] = 'kg_' + uuid.uuid4().hex[:12]
os.environ['LOAD_TEST_DATA'] = str(WORK / ('accounts-' + RUN_ID + '.json'))
origin = ('kaggle-cpu' if Path('/kaggle/working').is_dir()
          and os.environ.get('KAGGLE_KERNEL_RUN_TYPE') and platform.system() == 'Linux'
          else 'unverified-notebook')
os.environ['LOAD_TEST_ORIGIN'] = origin
metadata = {'backend': BACKEND_DESCRIPTION, 'source_revision': SOURCE_REVISION,
            'script_sha256': {name: hashlib.sha256((WORK / name).read_bytes()).hexdigest() for name in SCRIPTS},
            'generator_os': platform.platform(), 'python': platform.python_version(),
            'logical_cpus': os.cpu_count(), 'memory_bytes': psutil.virtual_memory().total,
            'cpu_model': platform.processor() or 'unknown',
            'cpu_confirmed_by_operator': CPU_CONFIRMED,
            'kaggle_kernel_run_type': os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'unknown'),
            'origin_declared': origin, 'approved_max_users': APPROVED_MAX_USERS,
            'created_at_utc': datetime.now(timezone.utc).isoformat()}
# Environment evidence lives separately so Locust output directory stays empty.
EVIDENCE = WORK / 'evidence' / OUTPUT.name
EVIDENCE.mkdir(parents=True, exist_ok=False)
(EVIDENCE / 'environment.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
shutil.copy2(WORK / 'connectivity.json', EVIDENCE / 'connectivity.json')
with (EVIDENCE / 'pip-freeze.txt').open('w') as handle:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=handle, check=True, timeout=30)
print('Origin tự khai báo:', origin, '; cần review bằng chứng trước VERIFIED.')


## Prepare → chạy đúng một profile

Prepare nằm ngoài thời gian đo, tạo tài khoản/giỏ riêng. Không chạy song song cùng manifest. Smoke phải thành công trước low; low thành công và kiểm tra tài nguyên backend trước medium/high. Nếu lỗi, vẫn chạy cell export để giữ artifacts; không coi run lỗi là PASS.

In [ ]:
subprocess.run([sys.executable, str(WORK / 'prepare.py'), '--users', str(users),
                '--output', os.environ['LOAD_TEST_DATA']], check=True, timeout=1200)
# Runner tự lưu execution.json khi timeout/fail; không bọc thêm timeout ngắn hơn ở notebook.
result = subprocess.run([sys.executable, str(WORK / 'run.py'), '--profile', PROFILE,
                         '--output', str(OUTPUT)], check=False)
print('Locust exit code:', result.returncode)
if result.returncode:
    print('Run FAILED/INCOMPLETE. Không tăng tải; tiếp tục export để điều tra.')
else:
    print('Run EXECUTED; chưa VERIFIED nguồn Kaggle CPU.')


In [ ]:
# Tổng hợp từ artifacts thật, kiểm tra counts/latency khớp nhau; không ghi đè summary.
ANALYSIS = WORK / 'analysis'
subprocess.run([sys.executable, str(WORK / 'report.py'), '--results', str(WORK / 'results'),
                '--output', str(ANALYSIS)], check=False, timeout=30)
report = json.loads((ANALYSIS / 'comparison.json').read_text(encoding='utf-8'))
for row in report['runs']:
    print({key: row[key] for key in ['run', 'execution', 'origin_declared', 'configured_users',
          'peak_observed_users', 'duration_seconds', 'total_requests', 'rps',
          'average_latency_ms', 'p95_latency_ms', 'p99_latency_ms', 'failure_rate', 'cache_hit_miss']})
print('Invalid/incomplete runs:', report['invalid_runs'])
print('Cache hit/miss NOT MEASURED: không suy từ latency hoặc Redis-wide counters.')


In [ ]:
import csv
import matplotlib.pyplot as plt
history_file = OUTPUT / 'locust_stats_history.csv'
if history_file.exists():
    with history_file.open() as handle:
        aggregate = [row for row in csv.DictReader(handle) if row['Name'] == 'Aggregated']
    if aggregate:
        start = float(aggregate[0]['Timestamp'])
        seconds = [float(row['Timestamp']) - start for row in aggregate]
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(seconds, [float(row['Requests/s']) for row in aggregate])
        axes[0].set(xlabel='Thời gian (s)', ylabel='Requests/s', title='RPS quan sát')
        axes[1].plot(seconds, [float(row['User Count']) for row in aggregate])
        axes[1].set(xlabel='Thời gian (s)', ylabel='Users', title='Concurrency quan sát')
        fig.tight_layout()
        fig.savefig(OUTPUT / 'metrics.png', dpi=150)
        plt.show()
else:
    print('Không có CSV history; không vẽ dữ liệu giả.')


In [ ]:
# Export mới mỗi lần, tránh gộp artifacts cũ. Không copy accounts, .env hoặc notebook secret.
EXPORT = WORK / ('export-' + RUN_ID)
EXPORT.mkdir(exist_ok=False)
for folder in ['results', 'evidence', 'analysis']:
    if (WORK / folder).exists():
        shutil.copytree(WORK / folder, EXPORT / folder)
archive = shutil.make_archive(str(WORK / ('cart-load-report-' + RUN_ID)), 'zip', EXPORT)
print('Tải file trong /kaggle/working:', archive)
# Không public ZIP trước khi rà soát URL, environment metadata và logs.


## Handoff

CSV/HTML trong mỗi run và comparison CSV/HTML/JSON là dữ liệu thực, không tự chứng minh đã chạy Kaggle. Review evidence CPU/runtime, revision, target được duyệt, execution.json, status counts, lỗi và thời lượng trước khi ghi VERIFIED trong báo cáo. Cache hit/miss vẫn NOT MEASURED. Khi chưa có backend HTTPS hoặc Kaggle session: **NOT RUN**.

[Kaggle Notebook settings](https://www.kaggle.com/docs/notebooks) · [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md)